# Big Data Foundation – Day 3
# Unstructured Data Analytics with PySpark

This Colab demonstrates how raw customer-review text can be processed with Spark.

**Flow:** Raw Text → Cleaning → Tokenization → MapReduce → Sentiment → Business Insights


In [ ]:
!pip install -q pyspark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random, re

spark = (SparkSession.builder
         .appName("UnstructuredDataAnalytics")
         .master("local[*]")
         .getOrCreate())

print("Spark version:", spark.version)


## 1. Structured vs Unstructured Data

**Structured:** fixed rows and columns, such as `Customer | Product | Rating`.

**Unstructured:** free-form content such as customer reviews, emails, social posts, documents, images, audio and video.

Today we use customer reviews because the useful information is hidden inside text.


In [ ]:
# Generate 5,000 raw customer reviews
positive = [
    "I absolutely love this product, the quality is excellent",
    "Amazing product and very good quality",
    "The product is fantastic and works perfectly",
    "Excellent quality and fast delivery",
    "Very happy with my purchase",
    "Great product, I highly recommend it",
    "The battery is excellent and lasts very long",
    "Amazing experience, the product is worth the money"
]

negative = [
    "I hate this product, the quality is terrible",
    "Very poor quality and bad experience",
    "The product stopped working after one day",
    "Delivery was very late and disappointing",
    "Terrible product, I do not recommend it",
    "The battery is poor and does not last",
    "Very unhappy with my purchase",
    "Bad quality and poor performance"
]

neutral = [
    "The product arrived yesterday",
    "The product is okay for the price",
    "Delivery took three days",
    "The product looks like the picture",
    "I received the product today",
    "The product has basic features",
    "Packaging was normal",
    "It works as described"
]

reviews = []
for _ in range(5000):
    reviews.append(random.choice([positive, negative, neutral])[0] if False else
                   random.choice(random.choice([positive, negative, neutral])))

print("Total reviews:", len(reviews))
for x in reviews[:10]:
    print("-", x)


## 2. Put the raw text into an RDD
Each review becomes an element of a Spark RDD. This connects directly to the distributed-processing concepts from Hadoop.


In [ ]:
review_rdd = spark.sparkContext.parallelize(reviews)

print("Reviews:", review_rdd.count())
print("Partitions:", review_rdd.getNumPartitions())


## 3. Clean the text
We convert to lowercase and remove punctuation so that words can be analyzed consistently.


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

cleaned_rdd = review_rdd.map(clean_text)

print("Original:", reviews[0])
print("Cleaned :", cleaned_rdd.first())


## 4. Tokenization
A sentence is split into individual words.

`excellent product and fast delivery` → `excellent`, `product`, `and`, `fast`, `delivery`


In [ ]:
tokens = cleaned_rdd.flatMap(lambda x: x.split())

print(tokens.take(30))


## 5. Remove stop words
Common words such as `the`, `is`, `and`, `to` usually contribute little to a simple word-frequency analysis.


In [ ]:
stop_words = {
    "the","is","and","a","of","to","it","this","was","for","my",
    "i","with","very","are","in","on","as","an","am","do","not","does"
}

filtered = tokens.filter(lambda w: w not in stop_words and len(w) > 2)

print(filtered.take(40))


## 6. MapReduce Word Frequency

**Map:** `word → (word, 1)`

**Reduce:** combine identical keys: `(word, 1) + (word, 1) + ...`


In [ ]:
word_counts = (filtered
               .map(lambda word: (word, 1))
               .reduceByKey(lambda a, b: a + b)
               .sortBy(lambda x: x[1], ascending=False))

print("Top 20 words:")
for word, count in word_counts.take(20):
    print(f"{word:15} {count}")


## 7. Simple Sentiment Analysis
This classroom demo uses a small positive/negative word dictionary. It is intentionally simple; production sentiment systems would normally use ML/NLP models.


In [ ]:
positive_words = {
    "excellent","amazing","good","great","love","happy",
    "fantastic","satisfied","recommend","perfectly"
}
negative_words = {
    "poor","bad","terrible","hate","disappointing",
    "defective","useless","unhappy","waste"
}

def sentiment(text):
    words = clean_text(text).split()
    p = sum(w in positive_words for w in words)
    n = sum(w in negative_words for w in words)
    return "Positive" if p > n else "Negative" if n > p else "Neutral"

for x in reviews[:10]:
    print(f"{sentiment(x):8} -> {x}")


## 8. Convert the unstructured text into a Spark DataFrame
This shows an important idea: unstructured data can be transformed into structured information for analytics.


In [ ]:
rows = [(i + 1, text, sentiment(text)) for i, text in enumerate(reviews)]

review_df = spark.createDataFrame(
    rows, ["Review_ID", "Review_Text", "Sentiment"]
)

review_df.show(10, truncate=False)


In [ ]:
# Sentiment distribution
summary = review_df.groupBy("Sentiment").count().orderBy(desc("count"))
summary.show()

total = review_df.count()
summary_pd = summary.toPandas()
summary_pd["Percentage"] = (summary_pd["count"] / total * 100).round(2)
summary_pd


## 9. Visualization – Customer Sentiment


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.bar(summary_pd["Sentiment"], summary_pd["count"])
plt.title("Customer Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()


## 10. Most Common Words


In [ ]:
top_words_df = spark.createDataFrame(
    word_counts.take(15), ["Word", "Frequency"]
)
top_words_df.show(15)

top_pd = top_words_df.toPandas()
plt.figure(figsize=(10,5))
plt.bar(top_pd["Word"], top_pd["Frequency"])
plt.title("Most Frequent Words in Customer Reviews")
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


# 11. Connect the Lab to Big Data

```text
Customer Reviews / Emails / Social Posts
                 ↓
             Raw Text
                 ↓
          Apache Spark RDD
                 ↓
        Clean + Tokenize
                 ↓
       Map → Shuffle → Reduce
                 ↓
          Spark DataFrame
                 ↓
        Sentiment / Analytics
                 ↓
          Business Insight
```

The important message for students: **Big Data is not only large structured tables. Large volumes of unstructured text also need scalable processing.**


# 12. Student Mini Challenge

1. Find the top 10 words.
2. Count reviews containing `quality`.
3. Find the percentage of negative reviews.
4. Find the most common words only in negative reviews.
5. Create a sentiment chart.

Hint for #4:
```python
review_df.filter(col("Sentiment") == "Negative")
```


In [ ]:
# Challenge solutions
print("1. Top 10 words")
for w, c in word_counts.take(10):
    print(w, c)

print("\n2. Reviews containing 'quality'")
print(review_df.filter(lower(col("Review_Text")).contains("quality")).count())

print("\n3. Negative percentage")
negative_count = review_df.filter(col("Sentiment") == "Negative").count()
print(round(negative_count / total * 100, 2), "%")

print("\n4. Common words in negative reviews")
negative_tokens = (review_df.filter(col("Sentiment") == "Negative")
                   .select("Review_Text").rdd
                   .flatMap(lambda r: clean_text(r["Review_Text"]).split())
                   .filter(lambda w: w not in stop_words and len(w) > 2))

negative_counts = (negative_tokens.map(lambda w: (w,1))
                   .reduceByKey(lambda a,b: a+b)
                   .sortBy(lambda x: x[1], ascending=False))

for w, c in negative_counts.take(10):
    print(w, c)


# Key Takeaways

- **Structured data:** fixed schema and columns.
- **Unstructured data:** free-form text, documents, images, audio, video.
- Spark can process large-scale unstructured text.
- RDD operations demonstrate MapReduce concepts.
- Unstructured data can be transformed into structured information and then analyzed.

**Production examples:** customer reviews, emails, social media, cybersecurity logs, enterprise documents and chat data.


In [ ]:
spark.stop()
print("Spark session stopped.")
